# Solution B3 · Custom features inside the pipeline, and saving the model for production

**Type**: Implement with scikit-learn

## Objective

Add computed features (family size, logarithmic fare) to the pipeline with `FunctionTransformer`, save
the whole pipeline with `joblib`, reload it and use it on a **single request** the way a production service would.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = ds.titanic()
X, y = df.drop(columns="survived"), df["survived"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
NUM = ["age", "siblings_spouses", "parents_children", "fare"]
CAT = ["sex", "port", "deck", "title"]
cv = StratifiedKFold(5, shuffle=True, random_state=0)

import tempfile
from pathlib import Path

import joblib
from sklearn.preprocessing import FunctionTransformer

path = Path(tempfile.mkdtemp()) / "titanic_model.joblib"
request = pd.DataFrame([{"pclass": 3, "sex": "male", "age": 22.0, "siblings_spouses": 1, "parents_children": 0,
                         "fare": 7.25, "port": "S", "deck": None, "title": "Mr"}])

## Commented solution

In [ ]:
def add_features(df):
    d = df.copy()
    d["family_size"] = d["siblings_spouses"] + d["parents_children"] + 1
    d["alone"] = (d["family_size"] == 1).astype(int)
    d["log_fare"] = np.log1p(d["fare"])
    return d

In [ ]:
pipe = make_pipeline(
    FunctionTransformer(add_features),
    ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUM + ["family_size", "log_fare"]),
        ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")), CAT),
        ("pass", "passthrough", ["pclass", "alone"]),
    ], sparse_threshold=0),
    LogisticRegression(max_iter=2000),
)
pipe.fit(X_tr, y_tr)
test_auc = roc_auc_score(y_te, pipe.predict_proba(X_te)[:, 1])
names = list(pipe[1].get_feature_names_out())
print(f"test AUC: {test_auc:.3f} | columns entering the model: {len(names)}")

In [ ]:
joblib.dump(pipe, path)
loaded = joblib.load(path)                                       # in production: another process, another machine
single_p = loaded.predict_proba(request)[0, 1]
identical = bool(np.isclose(single_p, pipe.predict_proba(request)[0, 1]))
print(f"survival probability for the request: {single_p:.3f} | equal to the original: {identical}")

## Check

In [ ]:
with verify():
    small = pd.DataFrame({"siblings_spouses": [1, 0], "parents_children": [2, 0], "fare": [10.0, 0.0]})
    out = add_features(small)
    check(list(out["family_size"]) == [4, 1] and list(out["alone"]) == [0, 1] and close(out["log_fare"], np.log1p([10.0, 0.0])),
          "add_features computes the three columns", "family_size = siblings + parents + 1; alone = family_size == 1; log_fare = log1p(fare)")
    check("family_size" not in small.columns, "the input is not modified in place", "use df.copy()")
    check(0.80 < test_auc < 0.93, "reasonable test AUC", "test_auc = AUC on the test set of the trained pipeline")
    check("num__family_size" in names and "num__log_fare" in names and any(n.endswith("alone") for n in names),
          "the new features reach the model", "include family_size and log_fare among the numerics and alone in passthrough")
    check(path.exists() and path.stat().st_size > 0, "the joblib file exists", "joblib.dump(pipe, path)")
    check(0 <= float(single_p) <= 1, "single_p is a probability", "predict_proba(request)[0, 1]")
    check(bool(identical) and close(single_p, pipe.predict_proba(request)[0, 1]), "the reloaded pipeline gives the same result as the original",
          "compare with pipe.predict_proba(request)")
done()

## What you learned

- The features computed inside the pipeline (`FunctionTransformer`) travel with the model: in production you pass the **raw**
  request and the pipeline does everything. Writing the computation twice (in training and in the service) is the first cause of
  training-serving skew.
- `joblib.dump` saves the whole pipeline (learned preprocessing included). On loading you need the **same version** of scikit-learn and
  the function code (`add_features`) importable: it is a real limitation, to document next to the file.
- A single request is a DataFrame with one row and the raw columns: it is the simplest test to discover hidden dependencies
  (for example a column that was always present in training).

## Common pitfall

Transformation functions that modify the DataFrame in place. In a pipeline the input DataFrame could be the same object
used elsewhere: `df.copy()` at the start avoids side effects that are hard to find.

## Interview question

> *Click on a question to show its answer.*

<details><summary>What do you save to put a model in production, and what can break on loading?</summary>

I save the whole pipeline (preprocessing and model) with `joblib`, together with the version of Python and of the libraries and a small
set of example requests with their predictions, to use as a regression test on loading. It can break because of the
scikit-learn version (the internal format changes), because of custom functions not importable in the service, or because of input
columns different from those of the training set.

</details>